In [ ]:
import pandas as pd
import numpy as np

# Define file paths
fam_file = 'C:/Users/mohak/Desktop/Data/2.3/rice_qc3_maf_filter.fam'
# THIS IS THE CORRECT FILE FOR THE PHENOTYPE DATA SHEET
phenotype_csv = 'C:/Users/mohak/Desktop/Data/3kRG_PhenotypeData_v20170411.xlsx' 
output_pheno_file = 'C:/Users/mohak/Desktop/awpr_repro_pheno.txt'

# Function to extract GWAS ID from the 'NAME' column
def extract_gwas_id(name_str):
    if pd.isna(name_str):
        return None
    name_str = str(name_str).strip()
    if '::' in name_str:
        id_part = name_str.split('::')[-1].strip()
        if id_part.startswith('IRGC '):
            return 'IRIS_' + id_part[5:].replace(' ', '_')
        else:
            return id_part
    return name_str.replace(' ', '_')

# Load genotype IDs (FID, IID) from .fam file
df_genotype_ids = pd.read_csv(fam_file, sep='\s+', header=None, usecols=[0, 1],
                              names=['FID', 'IID_genotype'])

# Load phenotype data and create Mapped_IID
df_pheno = pd.read_csv(phenotype_csv)
df_pheno['Mapped_IID'] = df_pheno['NAME'].apply(extract_gwas_id)

# Merge genotype IDs with phenotype data based on mapped IDs
df_merged = pd.merge(df_genotype_ids,
                     df_pheno[['Mapped_IID', 'AWPR_REPRO']],
                     left_on='IID_genotype', right_on='Mapped_IID',
                     how='left')

# Drop the duplicate mapping column and rename IID
df_merged.drop(columns=['Mapped_IID'], inplace=True)
df_merged.rename(columns={'IID_genotype': 'IID'}, inplace=True)

# Fill missing AWPR_REPRO values with -9 and convert to integer
df_merged['AWPR_REPRO'] = df_merged['AWPR_REPRO'].fillna(-9).astype(int)

# Select final columns and save to text file
df_final_pheno = df_merged[['FID', 'IID', 'AWPR_REPRO']]
df_final_pheno.to_csv(output_pheno_file, sep='\t', index=False, header=False)

print(f"Successfully created phenotype file: '{output_pheno_file}'")
print("First 5 rows of the generated phenotype file:")
print(df_final_pheno.head())
print(f"Total individuals in phenotype file: {len(df_final_pheno)}")
print(f"Individuals with missing AWPR_REPRO data (coded as -9): {(df_final_pheno['AWPR_REPRO'] == -9).sum()}")

In [3]:
import pandas as pd

# Define the path to your phenotype CSV file
phenotype_csv = 'C:/Users/mohak/Desktop/Data/3kRG_PhenotypeData_v20170411.xlsx'

# Load the raw phenotype data
df_raw_pheno = pd.read_excel(phenotype_csv)

# Print all detected column names
print("Columns in your phenotype data file:")
print(df_raw_pheno.columns.tolist())

Columns in your phenotype data file:
['WORKSHEET', 'REMARKS']


c:\Users\mohak\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
